<a href="https://colab.research.google.com/github/NANInithin/Vehicle-speed-estimation-through-aerial-videos/blob/main/Speed_estimation_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ultralytics
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from collections import defaultdict
from google.colab.patches import cv2_imshow
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.2/915.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [18]:
fx, fy, cx, cy = 500.0, 500.0, 320.0, 240.0
k1, k2, p1, p2, k3 = 0, 0, 0, 0, 0
camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
distortion_coeffs = np.array([k1, k2, p1, p2, k3])

H = np.array([[1.000, 2.623e-13, 7.5000],
              [-9.59e-14, 1.000, 7.5000],
               [-1.27e-16, -1.34e-16, 1.000]])

def pixel_to_meter(x, y, H):
  point = np.array([[x], [y], [1]])
  world_point = np.dot(H, point)
  world_point = world_point / world_point[2]
  return world_point[0], world_point[1]

def moving_average(data, window_size):
    window = np.ones(window_size) / window_size
    return np.convolve(data, window, 'same')

def interpolate(data, timestamps):
  data_interpolated = []
  timestamps_interpolated = np.arange(timestamps[0], timestamps[-1], 0.1)
  for i in range(len(data[0])):
    series = [point[i] for point in data]
    series_interpolated = np.interp(timestamps_interpolated, timestamps, series)
    data_interpolated.append(series_interpolated)
  return list(zip(*data_interpolated)), timestamps_interpolated

model_path = "/content/drive/MyDrive/Project/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

video_path = "/content/drive/MyDrive/Project/istockphoto-1461871231-640_adpp_is.mp4"
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error opening video file")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_video_path = '/content/drive/MyDrive/Project/output_video.mp4'
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
drone_height = 45

fov_degrees = 30
fov_radians = np.deg2rad(fov_degrees)

visible_width = 2 * np.tan(fov_radians / 2) * drone_height
conversion_factor = visible_width / frame_width

vehicle_data = defaultdict(lambda: {'timestamps': [], 'positions': [], 'speeds': [], 'accelations': []})

vehicle_classes = {3, 4, 5, 8}

while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    break
  frame_number = cap.get(cv2.CAP_PROP_POS_FRAMES)
  results = model.track(frame, persist=True, tracker = 'botsort.yaml')

  for result in results:
    if hasattr(result, 'plot'):
      frame_with_boxes = result.plot()
      if frame_with_boxes is not None:
        out.write(frame_with_boxes)
      for box in result.boxes:
        if int(box.cls[0]) in vehicle_classes:
          if box.id is not None:
            obj_id = int(box.id[0])
          else:
            continue
          x, y, w, h = box.xywh.cpu().numpy().flatten() * conversion_factor
          timestamp = frame_number / fps
          position = (x, y)

          vehicle_data[obj_id]['timestamps'].append(timestamp)
          vehicle_data[obj_id]['positions'].append(position)

cap.release()
out.release()
cv2.destroyAllWindows()

window_size = 5

for obj_id, info in vehicle_data.items():
  if len(info['positions']) >= window_size:
    positions, timestamps = interpolate(info['positions'], info['timestamps'])
    positions_x = [pos[0] for pos in positions]
    positions_y = [pos[1] for pos in positions]
    smooth_positions_x = moving_average(positions_x, window_size)
    smooth_positions_y = moving_average(positions_y, window_size)

    info['positions'] = list(zip(smooth_positions_x, smooth_positions_y))
    info['timestamps'] = timestamps[:len(smooth_positions_x)]


for obj_id, info in vehicle_data.items():
  for i in range(1, len(info['positions'])): # Start from index 1 to avoid accessing index -1
    # if i == 0:  # This condition is redundant as i starts from 1
    #  info['speeds'].append((0, 0))
    #  info['accelations'].append((0, 0))
    #else:
    # Access timestamps based on the current smoothed positions length
    try: #Using a try and except block to catch index errors
      dt = info['timestamps'][i] - info['timestamps'][i - 1]
    except IndexError:
      #If index error, append 0 and continue to next iteration
      info['speeds'].append(0)
      info['accelations'].append(0)
      continue

    dx = info['positions'][i][0] - info['positions'][i - 1][0]
    dy = info['positions'][i][1] - info['positions'][i - 1][1]

    if dt > 0:
      speed = (dx**2 + dy**2)**0.5 / dt  # Calculating speed using both dx and dy
      info['speeds'].append(speed)

      if i > 1:
        prev_speed = info['speeds'][i - 2] # Access previous speed from correct index
        acceleration = (speed - prev_speed) / dt
        info['accelations'].append(acceleration)
    else:
      info['speeds'].append(0)
      info['accelations'].append(0)

fx, fy, cx, cy = 500.0, 500.0, 320.0, 240.0
k1, k2, p1, p2, k3 = 0, 0, 0, 0, 0
camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
distortion_coeffs = np.array([k1, k2, p1, p2, k3])

H = np.array([[1.000, 2.623e-13, 7.5000],
              [-9.59e-14, 1.000, 7.5000],
               [-1.27e-16, -1.34e-16, 1.000]])

def pixel_to_meter(x, y, H):
  point = np.array([[x], [y], [1]])
  world_point = np.dot(H, point)
  world_point = world_point / world_point[2]
  return world_point[0], world_point[1]

def moving_average(data, window_size):
    window = np.ones(window_size) / window_size
    return np.convolve(data, window, 'same')

def interpolate(data, timestamps):
  data_interpolated = []
  timestamps_interpolated = np.arange(timestamps[0], timestamps[-1], 0.1)
  for i in range(len(data[0])):
    series = [point[i] for point in data]
    series_interpolated = np.interp(timestamps_interpolated, timestamps, series)
    data_interpolated.append(series_interpolated)
  return list(zip(*data_interpolated)), timestamps_interpolated

model_path = "/content/drive/MyDrive/Project/runs/detect/train/weights/best.pt"
model = YOLO(model_path)

video_path = "/content/drive/MyDrive/Project/istockphoto-1357370333-640_adpp_is.mp4"
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Error opening video file")
    exit()

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

drone_height = 45

fov_degrees = 30
fov_radians = np.deg2rad(fov_degrees)

visible_width = 2 * np.tan(fov_radians / 2) * drone_height
conversion_factor = visible_width / frame_width

vehicle_data = defaultdict(lambda: {'timestamps': [], 'positions': [], 'speeds': [], 'accelations': []})

vehicle_classes = {3, 4, 5, 8}

# Assuming 'out' is defined elsewhere (e.g., video writer)
# out = cv2.VideoWriter(...)

while cap.isOpened():
  ret, frame = cap.read()
  if not ret:
    break
  frame_number = cap.get(cv2.CAP_PROP_POS_FRAMES)
  results = model.track(frame, persist=True, tracker = 'botsort.yaml')

  for result in results:
    if hasattr(result, 'plot'):
      frame_with_boxes = result.plot()
      if frame_with_boxes is not None:
        out.write(frame_with_boxes)
      for box in result.boxes:
        if int(box.cls[0]) in vehicle_classes:
          if box.id is not None:
            obj_id = int(box.id[0])
          else:
            continue
          x, y, w, h = box.xywh.cpu().numpy().flatten() * conversion_factor
          timestamp = frame_number / fps
          position = (x, y)

          vehicle_data[obj_id]['timestamps'].append(timestamp)
          vehicle_data[obj_id]['positions'].append(position)

cap.release()
out.release()
cv2.destroyAllWindows()

window_size = 5

for obj_id, info in vehicle_data.items():
  if len(info['positions']) >= window_size:
    positions, timestamps = interpolate(info['positions'], info['timestamps'])
    positions_x = [pos[0] for pos in positions]
    positions_y = [pos[1] for pos in positions]
    smooth_positions_x = moving_average(positions_x, window_size)
    smooth_positions_y = moving_average(positions_y, window_size)

    info['positions'] = list(zip(smooth_positions_x, smooth_positions_y))
    info['timestamps'] = timestamps[:len(smooth_positions_x)]

Streaming output truncated to the last 5000 lines.
0: 384x640 13 cars, 2 trucks, 2 buss, 9.6ms
Speed: 1.8ms preprocess, 9.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 3 trucks, 1 bus, 9.5ms
Speed: 1.9ms preprocess, 9.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 3 trucks, 1 bus, 9.5ms
Speed: 1.9ms preprocess, 9.5ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 3 trucks, 1 bus, 9.0ms
Speed: 2.0ms preprocess, 9.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 13 cars, 3 trucks, 1 bus, 9.0ms
Speed: 1.6ms preprocess, 9.0ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 3 trucks, 1 bus, 9.6ms
Speed: 2.3ms preprocess, 9.6ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 3 trucks, 1 bus, 9.4ms
Speed: 1.9ms preprocess, 9.4ms inference, 1.3ms postprocess per i

In [17]:
for obj_id, info in vehicle_data.items():
  info['speeds'] = []  # Reset speeds list for each object
  info['accelations'] = []  # Reset accelerations list for each object
  # Iterate based on the length of timestamps, to be consistent
  for i in range(1, len(info['timestamps'])):
    dt = info['timestamps'][i] - info['timestamps'][i - 1]
    # Access positions based on the current index of timestamps
    dx = info['positions'][i][0] - info['positions'][i - 1][0]
    dy = info['positions'][i][1] - info['positions'][i - 1][1]

    if dt > 0:
      speed = (dx**2 + dy**2)**0.5 / dt
      info['speeds'].append(speed)

      if i > 1: # Make sure we have enough data to calculate acceleration
          try:
              prev_speed = info['speeds'][i - 2] # Access speed from the previous frame
              acceleration = (speed - prev_speed) / dt
              info['accelations'].append(acceleration)
          except IndexError:
              # Handles cases when there may be less data points available
              info['accelations'].append(0) #Append 0 if it tries to access negative index
    else:
      info['speeds'].append(0)
      info['accelations'].append(0)

# Assuming 'files' is defined elsewhere (e.g., from google.colab import files)
# files.download('/content/drive/MyDrive/project/vehicle_data.csv')
data = []
for obj_id, info in vehicle_data.items():
    # Ensure that you iterate only up to the minimum length of all relevant lists
    min_len = min(len(info['timestamps']), len(info['positions']), len(info['speeds']), len(info['accelations']))
    for i in range(min_len):
        data.append([obj_id, info['timestamps'][i], info['positions'][i][0], info['positions'][i][1], info['speeds'][i], info['accelations'][i]])

df = pd.DataFrame(data, columns=['obj_id', 'timestamp', 'x', 'y', 'speed', 'acceleration'])
df.to_csv('/content/drive/MyDrive/Project/vehicle_data.csv', index=False)
print(f"DataFrame saved to /content/drive/MyDrive/Project/vehicle_data.csv")


DataFrame saved to /content/drive/MyDrive/Project/vehicle_data.csv
